# Zaskaleta AI Twin — AUTO v3
One-click production: Google Drive → clone assets → next unfinished Day → voices → realistic scenes → lip-sync → final 9:16 MP4.

**From you:** Runtime with T4 GPU, then `Run all`, and approve Google Drive access once.


In [ ]:
import os, re, subprocess, sys
from pathlib import Path
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Увімкніть T4 GPU: Runtime → Change runtime type → T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

ROOT=Path('/content/zaskaleta-ai-twin-colab')
if ROOT.exists():
    subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git',str(ROOT)],check=True)

WORKER=ROOT/'worker'
env=os.environ.copy()
env['APP_DIR']=str(WORKER)
env['MUSETALK_ROOT']='/content/MuseTalk'
env['VENV_DIR']='/content/ai-twin-py311'
subprocess.run(['bash',str(WORKER/'install_gpu_engines.sh')],check=True,env=env)

PY='/content/ai-twin-py311/bin/python'
cmd=[PY,str(WORKER/'run_auto_v3.py'),'--root',str(ROOT),'--mydrive','/content/drive/MyDrive']
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
lines=[]
for line in proc.stdout:
    print(line,end='')
    lines.append(line)
code=proc.wait()
if code != 0:
    raise RuntimeError(f'AUTO v3 stopped with exit code {code}')
out=''.join(lines)
m=re.search(r'^FINAL_PATH=(.+)$',out,re.MULTILINE)
if not m:
    raise RuntimeError('AUTO v3 завершився без FINAL_PATH')
FINAL=m.group(1).strip()

from IPython.display import Video, display
print('✅ FINAL:',FINAL)
display(Video(FINAL,embed=True,width=360))
